# 03 Rule Classification

Apply deterministic first-pass rules to the latest inventory output and produce a review table. Still dry-run only.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

POLICY_PATH = PROJECT_ROOT / 'policy' / 'SCH_fileserver_policy_v2_4.yaml'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('POLICY_PATH =', POLICY_PATH)


In [ ]:
from datetime import datetime
import pandas as pd

from src.policy_loader import PolicyLoader
from src.rules import classify_inventory, save_rule_outputs

policy = PolicyLoader.from_file(POLICY_PATH)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
inventory_files = sorted(OUTPUT_DIR.glob('inventory_*.parquet'))
assert inventory_files, 'No inventory parquet files found. Run 02_inventory.ipynb first.'
latest_inventory = inventory_files[-1]
print('Using inventory:', latest_inventory.name)
inv = pd.read_parquet(latest_inventory)
print('Rows:', len(inv))
print('Columns:', list(inv.columns))


## Schema note
`classify_inventory()` now backfills missing inventory fields such as `filename`, `suffix`, `parent_relative`, `path_length`, and `filename_length` if you loaded an older inventory parquet. For best consistency, rerun `02_inventory.ipynb` after policy or scanner changes.


In [ ]:
classified = classify_inventory(inv, policy)
classified[['relative_path', 'rule_status', 'rule_reason', 'rule_confidence', 'proposed_relative_target']].head(20)

In [ ]:
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'rule_classification_{stamp}'
csv_path, parquet_path = save_rule_outputs(classified, output_base)
print('CSV:', csv_path)
print('Parquet:', parquet_path)


In [ ]:
classified.groupby('rule_status').size().sort_values(ascending=False).to_frame('count')

In [ ]:
classified[classified['rule_status'] == 'archive_or_delete_candidate'][['relative_path', 'filename', 'rule_reason', 'proposed_relative_target']].head(50)

In [ ]:
classified[classified['rule_status'] == 'move_to_special_folder'][['relative_path', 'filename', 'rule_reason', 'special_folder_target', 'proposed_relative_target']].head(50)

In [ ]:
classified[classified['rule_status'] == 'compliant_keep_review_path'][['relative_path', 'filename', 'parsed_phase', 'parsed_doc_type', 'default_folder_subpath', 'proposed_relative_target']].head(50)

In [ ]:
classified[classified['rule_status'] == 'review'][['relative_path', 'filename', 'rule_reason', 'path_risk', 'filename_risk']].head(50)

In [ ]:
classified.sort_values(['action_priority', 'relative_path']).head(100)